In [ ]:
''' KPIS:
1) Completion Rate test Vs control 
completed by id / total started by id
2) average time of iteration test vs control 
average (total time by user id/ total steps by user id)
3) Error Rate = number of errors / number of sessions or attempts
4) Step Completion Rate = clients who reached next step / clients who reached current step
5) Average number of step visits per client
average(total steps by client_id)
6)  Median Time per Step test vs control
    Average Time per Step
    Total Completion Time
    Higher completion rate + lower or similar time spent
7)Backtracking Rate = clients who move backwards / clients who started
8)Drop-off Rate = clients who abandon at step / clients who reached that step

9) Client Segment Performance:
Age group
Gender
Tenure
Balance group
Number of accounts
Calls/logons activity '''

In [4]:
import pandas as pd
import numpy as np

# =========================
# 1. Completion Rate KPI
# =========================

kpi_df = web_df[web_df["Variation"].isin(["Test", "Control"])].copy()

kpi_df["started"] = kpi_df["process_step"].eq("start")
kpi_df["completed"] = kpi_df["process_step"].eq("confirm")

client_completion = (
    kpi_df.groupby(["Variation", "client_id"], as_index=False)
    .agg(started=("started", "max"), completed=("completed", "max"))
)

client_completion["started_and_completed"] = client_completion["started"] & client_completion["completed"]

completion_rate_kpi = (
    client_completion.groupby("Variation")
    .agg(
        total_started_clients=("started", "sum"),
        completed_clients=("started_and_completed", "sum")
    )
    .reset_index()
)

completion_rate_kpi["completion_rate_%"] = (
    completion_rate_kpi["completed_clients"] / completion_rate_kpi["total_started_clients"] * 100
).round(2)

completion_rate_kpi

,Variation,total_started_clients,completed_clients,completion_rate_%
0,Control,23397,15318,65.47
1,Test,26679,18412,69.01


In [5]:
# =========================
# 2. Average Iteration Time KPI
# =========================

kpi_df = web_df[web_df["Variation"].isin(["Test", "Control"])].copy()
kpi_df["date_time"] = pd.to_datetime(kpi_df["date_time"])

# Visit-level summary
visit_summary = (
    kpi_df.groupby(["Variation", "client_id", "visit_id"], as_index=False)
    .agg(
        visit_start=("date_time", "min"),
        visit_end=("date_time", "max"),
        total_steps=("process_step", "count")
    )
)

visit_summary["visit_duration_seconds"] = (
    visit_summary["visit_end"] - visit_summary["visit_start"]
).dt.total_seconds()

# Client-level summary
client_iteration_time = (
    visit_summary.groupby(["Variation", "client_id"], as_index=False)
    .agg(
        total_time_seconds=("visit_duration_seconds", "sum"),
        total_steps=("total_steps", "sum")
    )
)

client_iteration_time["avg_iteration_time_seconds_by_client"] = (
    client_iteration_time["total_time_seconds"] / client_iteration_time["total_steps"]
)

# Normal KPI
avg_iteration_time_kpi = (
    client_iteration_time.groupby("Variation")
    .agg(
        clients=("client_id", "nunique"),
        total_steps=("total_steps", "sum"),
        avg_iteration_time_seconds=("avg_iteration_time_seconds_by_client", "mean"),
        median_iteration_time_seconds=("avg_iteration_time_seconds_by_client", "median")
    )
    .reset_index()
)

avg_iteration_time_kpi["avg_iteration_time_seconds"] = avg_iteration_time_kpi["avg_iteration_time_seconds"].round(2)
avg_iteration_time_kpi["median_iteration_time_seconds"] = avg_iteration_time_kpi["median_iteration_time_seconds"].round(2)

avg_iteration_time_kpi

,Variation,clients,total_steps,avg_iteration_time_seconds,median_iteration_time_seconds
0,Control,23532,140536,54.50,39.0
1,Test,26968,176699,56.66,38.6


In [6]:
# =========================
# 2B. Average Iteration Time KPI - Trimmed 80%
# =========================

q_bounds = (
    client_iteration_time.groupby("Variation")["avg_iteration_time_seconds_by_client"]
    .quantile([0.10, 0.90])
    .unstack()
    .reset_index()
    .rename(columns={0.10: "q10", 0.90: "q90"})
)

client_iteration_time_trimmed = client_iteration_time.merge(q_bounds, on="Variation", how="left")

client_iteration_time_trimmed = client_iteration_time_trimmed[
    client_iteration_time_trimmed["avg_iteration_time_seconds_by_client"].between(
        client_iteration_time_trimmed["q10"],
        client_iteration_time_trimmed["q90"]
    )
].copy()

avg_iteration_time_trimmed_kpi = (
    client_iteration_time_trimmed.groupby("Variation")
    .agg(
        clients_included=("client_id", "nunique"),
        total_steps=("total_steps", "sum"),
        avg_iteration_time_seconds_trimmed=("avg_iteration_time_seconds_by_client", "mean"),
        median_iteration_time_seconds_trimmed=("avg_iteration_time_seconds_by_client", "median")
    )
    .reset_index()
)

avg_iteration_time_trimmed_kpi["avg_iteration_time_seconds_trimmed"] = avg_iteration_time_trimmed_kpi["avg_iteration_time_seconds_trimmed"].round(2)
avg_iteration_time_trimmed_kpi["median_iteration_time_seconds_trimmed"] = avg_iteration_time_trimmed_kpi["median_iteration_time_seconds_trimmed"].round(2)

avg_iteration_time_trimmed_kpi

,Variation,clients_included,total_steps,avg_iteration_time_seconds_trimmed,median_iteration_time_seconds_trimmed
0,Control,21178,122885,39.17,35.33
1,Test,21574,148602,44.73,38.60


In [8]:
avg_iteration_comparison_kpi = avg_iteration_time_kpi.merge(
    avg_iteration_time_trimmed_kpi,
    on="Variation",
    how="left",
    suffixes=("_normal", "_trimmed")
)

avg_iteration_comparison_kpi["clients_removed"] = (
    avg_iteration_comparison_kpi["clients"] - avg_iteration_comparison_kpi["clients_included"]
)

avg_iteration_comparison_kpi["clients_removed_%"] = (
    avg_iteration_comparison_kpi["clients_removed"] / avg_iteration_comparison_kpi["clients"] * 100
).round(2)

avg_iteration_comparison_kpi

,Variation,clients,total_steps_normal,avg_iteration_time_seconds,median_iteration_time_seconds,clients_included,total_steps_trimmed,avg_iteration_time_seconds_trimmed,median_iteration_time_seconds_trimmed,clients_removed,clients_removed_%
0,Control,23532,140536,54.50,39.0,21178,122885,39.17,35.33,2354,10.0
1,Test,26968,176699,56.66,38.6,21574,148602,44.73,38.60,5394,20.0


In [ ]:
# =========================
# 3. Error/backward movement Rate KPI
# =========================

kpi_df = web_df[web_df["Variation"].isin(["Test", "Control"])].copy()
kpi_df["date_time"] = pd.to_datetime(kpi_df["date_time"])

# Define process order
step_order = {
    "start": 0,
    "step_1": 1,
    "step_2": 2,
    "step_3": 3,
    "confirm": 4
}

kpi_df["step_order"] = kpi_df["process_step"].map(step_order)

# Sort by user/session/time
kpi_df = kpi_df.sort_values(["Variation", "client_id", "visit_id", "date_time"])

# Previous step within each session
kpi_df["previous_step_order"] = (
    kpi_df.groupby(["Variation", "client_id", "visit_id"])["step_order"]
    .shift(1)
)

# Error = user moves backwards in the process
kpi_df["error_event"] = kpi_df["step_order"] < kpi_df["previous_step_order"]

# Session-level summary
session_errors = (
    kpi_df.groupby(["Variation", "client_id", "visit_id"], as_index=False)
    .agg(
        total_steps=("process_step", "count"),
        error_events=("error_event", "sum"),
        started=("process_step", lambda x: (x == "start").any())
    )
)

session_errors["had_error"] = session_errors["error_events"] > 0

# Final KPI table
error_rate_kpi = (
    session_errors.groupby("Variation")
    .agg(
        sessions=("visit_id", "nunique"),
        started_sessions=("started", "sum"),
        error_events=("error_events", "sum"),
        sessions_with_errors=("had_error", "sum")
    )
    .reset_index()
)

error_rate_kpi["error_rate_per_session"] = (
    error_rate_kpi["error_events"] / error_rate_kpi["sessions"]
).round(4)

error_rate_kpi["sessions_with_error_rate_%"] = (
    error_rate_kpi["sessions_with_errors"] / error_rate_kpi["sessions"] * 100
).round(2)

error_rate_kpi["error_rate_per_started_session"] = (
    error_rate_kpi["error_events"] / error_rate_kpi["started_sessions"]
).round(4)

error_rate_kpi

,Variation,sessions,started_sessions,error_events,sessions_with_errors,error_rate_per_session,sessions_with_error_rate_%,error_rate_per_started_session
0,Control,32189,30962,9581,6522,0.2976,20.26,0.3094
1,Test,37136,33219,16232,9969,0.4371,26.84,0.4886


In [10]:
error_rate_kpi_clean = error_rate_kpi[
    [
        "Variation",
        "sessions",
        "started_sessions",
        "error_events",
        "sessions_with_errors",
        "sessions_with_error_rate_%",
        "error_rate_per_session"
    ]
]

error_rate_kpi_clean

,Variation,sessions,started_sessions,error_events,sessions_with_errors,sessions_with_error_rate_%,error_rate_per_session
0,Control,32189,30962,9581,6522,20.26,0.2976
1,Test,37136,33219,16232,9969,26.84,0.4371


In [12]:
# =========================
# 4. Step Completion Rate KPI
# Table ordered by step, then Variation
# =========================

kpi_df = web_df[web_df["Variation"].isin(["Test", "Control"])].copy()

step_order = ["start", "step_1", "step_2", "step_3", "confirm"]
variation_order = ["Control", "Test"]

# Client-level step flags: one row per client and Variation
client_steps = (
    kpi_df.groupby(["Variation", "client_id"])["process_step"]
    .apply(set)
    .reset_index(name="steps_reached")
)

for step in step_order:
    client_steps[f"reached_{step}"] = client_steps["steps_reached"].apply(lambda x: step in x)

# Calculate step-to-step completion rates
step_completion_results = []

for current_step, next_step in zip(step_order[:-1], step_order[1:]):
    for variation in variation_order:
        temp = client_steps[client_steps["Variation"] == variation]

        clients_current_step = temp[f"reached_{current_step}"].sum()
        clients_next_step = temp[f"reached_{next_step}"].sum()

        step_completion_rate = clients_next_step / clients_current_step if clients_current_step > 0 else np.nan

        step_completion_results.append({
            "Variation": variation,
            "current_step": current_step,
            "next_step": next_step,
            "clients_reached_current_step": clients_current_step,
            "clients_reached_next_step": clients_next_step,
            "step_completion_rate_%": round(step_completion_rate * 100, 2),
            "drop_off_clients": clients_current_step - clients_next_step,
            "drop_off_rate_%": round((1 - step_completion_rate) * 100, 2)
        })

step_completion_kpi = pd.DataFrame(step_completion_results)

step_completion_kpi

,Variation,current_step,next_step,clients_reached_current_step,clients_reached_next_step,step_completion_rate_%,drop_off_clients,drop_off_rate_%
0,Control,start,step_1,23397,20152,86.13,3245,13.87
1,Test,start,step_1,26679,24267,90.96,2412,9.04
2,Control,step_1,step_2,20152,18650,92.55,1502,7.45
3,Test,step_1,step_2,24267,22258,91.72,2009,8.28
4,Control,step_2,step_3,18650,17422,93.42,1228,6.58
5,Test,step_2,step_3,22258,20881,93.81,1377,6.19
6,Control,step_3,confirm,17422,15434,88.59,1988,11.41
7,Test,step_3,confirm,20881,18687,89.49,2194,10.51


In [13]:
# =========================
# 5. Average Number of Step Visits per Client
# =========================

kpi_df = web_df[web_df["Variation"].isin(["Test", "Control"])].copy()

# Client-level summary
client_step_visits = (
    kpi_df.groupby(["Variation", "client_id"], as_index=False)
    .agg(
        step_visits=("process_step", "count"),
        unique_steps_reached=("process_step", "nunique"),
        sessions=("visit_id", "nunique")
    )
)

# Final KPI table
avg_step_visits_kpi = (
    client_step_visits.groupby("Variation")
    .agg(
        clients=("client_id", "nunique"),
        total_step_visits=("step_visits", "sum"),
        avg_step_visits_per_client=("step_visits", "mean"),
        median_step_visits_per_client=("step_visits", "median"),
        avg_unique_steps_reached=("unique_steps_reached", "mean"),
        avg_sessions_per_client=("sessions", "mean")
    )
    .reset_index()
)

avg_step_visits_kpi["avg_step_visits_per_client"] = avg_step_visits_kpi["avg_step_visits_per_client"].round(2)
avg_step_visits_kpi["median_step_visits_per_client"] = avg_step_visits_kpi["median_step_visits_per_client"].round(2)
avg_step_visits_kpi["avg_unique_steps_reached"] = avg_step_visits_kpi["avg_unique_steps_reached"].round(2)
avg_step_visits_kpi["avg_sessions_per_client"] = avg_step_visits_kpi["avg_sessions_per_client"].round(2)

avg_step_visits_kpi

,Variation,clients,total_step_visits,avg_step_visits_per_client,median_step_visits_per_client,avg_unique_steps_reached,avg_sessions_per_client
0,Control,23532,140536,5.97,5.0,4.04,1.37
1,Test,26968,176699,6.55,5.0,4.18,1.38


In [14]:
# =========================
# 5B. Average Number of Step Visits per Client - Trimmed 80%
# =========================

# Calculate 10th and 90th percentiles by Variation
q_bounds = (
    client_step_visits.groupby("Variation")["step_visits"]
    .quantile([0.10, 0.90])
    .unstack()
    .reset_index()
    .rename(columns={0.10: "q10", 0.90: "q90"})
)

# Add bounds to each client
client_step_visits_trimmed = client_step_visits.merge(q_bounds, on="Variation", how="left")

# Keep only central 80% of clients
client_step_visits_trimmed = client_step_visits_trimmed[
    client_step_visits_trimmed["step_visits"].between(
        client_step_visits_trimmed["q10"],
        client_step_visits_trimmed["q90"]
    )
].copy()

# Final trimmed KPI table
avg_step_visits_trimmed_kpi = (
    client_step_visits_trimmed.groupby("Variation")
    .agg(
        clients_included=("client_id", "nunique"),
        total_step_visits_trimmed=("step_visits", "sum"),
        avg_step_visits_per_client_trimmed=("step_visits", "mean"),
        median_step_visits_per_client_trimmed=("step_visits", "median")
    )
    .reset_index()
)

avg_step_visits_trimmed_kpi["avg_step_visits_per_client_trimmed"] = avg_step_visits_trimmed_kpi["avg_step_visits_per_client_trimmed"].round(2)
avg_step_visits_trimmed_kpi["median_step_visits_per_client_trimmed"] = avg_step_visits_trimmed_kpi["median_step_visits_per_client_trimmed"].round(2)

avg_step_visits_trimmed_kpi

,Variation,clients_included,total_step_visits_trimmed,avg_step_visits_per_client_trimmed,median_step_visits_per_client_trimmed
0,Control,19214,107402,5.59,5.0
1,Test,21886,133704,6.11,5.0


In [15]:
# =========================
# 6. Time Efficiency KPIs
# =========================

kpi_df = web_df[web_df["Variation"].isin(["Test", "Control"])].copy()
kpi_df["date_time"] = pd.to_datetime(kpi_df["date_time"])

step_order = ["start", "step_1", "step_2", "step_3", "confirm"]
variation_order = ["Control", "Test"]

kpi_df["process_step"] = pd.Categorical(kpi_df["process_step"], categories=step_order, ordered=True)

# Sort events chronologically within each session
kpi_df = kpi_df.sort_values(["Variation", "client_id", "visit_id", "date_time"])

In [16]:
# =========================
# 6A. Time per Step
# =========================

kpi_df["next_date_time"] = (
    kpi_df.groupby(["Variation", "client_id", "visit_id"])["date_time"]
    .shift(-1)
)

kpi_df["next_process_step"] = (
    kpi_df.groupby(["Variation", "client_id", "visit_id"])["process_step"]
    .shift(-1)
)

kpi_df["time_to_next_step_seconds"] = (
    kpi_df["next_date_time"] - kpi_df["date_time"]
).dt.total_seconds()

# Keep valid step transitions only
step_time_df = kpi_df[
    (kpi_df["time_to_next_step_seconds"].notna()) &
    (kpi_df["time_to_next_step_seconds"] >= 0) &
    (kpi_df["process_step"] != "confirm")
].copy()

time_per_step_kpi = (
    step_time_df.groupby(["process_step", "Variation"], observed=True)
    .agg(
        step_transitions=("time_to_next_step_seconds", "count"),
        avg_time_per_step_seconds=("time_to_next_step_seconds", "mean"),
        median_time_per_step_seconds=("time_to_next_step_seconds", "median")
    )
    .reset_index()
)

time_per_step_kpi["avg_time_per_step_seconds"] = time_per_step_kpi["avg_time_per_step_seconds"].round(2)
time_per_step_kpi["median_time_per_step_seconds"] = time_per_step_kpi["median_time_per_step_seconds"].round(2)

time_per_step_kpi["Variation"] = pd.Categorical(time_per_step_kpi["Variation"], categories=variation_order, ordered=True)
time_per_step_kpi["process_step"] = pd.Categorical(time_per_step_kpi["process_step"], categories=step_order[:-1], ordered=True)

time_per_step_kpi = time_per_step_kpi.sort_values(["process_step", "Variation"]).reset_index(drop=True)

time_per_step_kpi

,process_step,Variation,step_transitions,avg_time_per_step_seconds,median_time_per_step_seconds
0,start,Control,35737,66.83,20.0
1,start,Test,46323,61.47,14.0
2,step_1,Control,26043,50.47,20.0
3,step_1,Test,35527,60.68,27.0
4,step_2,Control,24313,92.00,64.0
5,step_2,Test,29573,88.85,61.0
6,step_3,Control,20280,137.14,72.0
7,step_3,Test,23983,129.64,58.0


In [17]:
# =========================
# 6B. Total Completion Time
# =========================

session_step_times = (
    kpi_df.groupby(["Variation", "client_id", "visit_id", "process_step"], observed=True)["date_time"]
    .min()
    .unstack()
    .reset_index()
)

completed_sessions_time = session_step_times[
    session_step_times["start"].notna() &
    session_step_times["confirm"].notna()
].copy()

completed_sessions_time["completion_time_seconds"] = (
    completed_sessions_time["confirm"] - completed_sessions_time["start"]
).dt.total_seconds()

completed_sessions_time = completed_sessions_time[
    completed_sessions_time["completion_time_seconds"] >= 0
].copy()

completed_sessions_time["completion_time_minutes"] = completed_sessions_time["completion_time_seconds"] / 60

total_completion_time_kpi = (
    completed_sessions_time.groupby("Variation")
    .agg(
        completed_sessions=("visit_id", "nunique"),
        avg_completion_time_seconds=("completion_time_seconds", "mean"),
        median_completion_time_seconds=("completion_time_seconds", "median"),
        avg_completion_time_minutes=("completion_time_minutes", "mean"),
        median_completion_time_minutes=("completion_time_minutes", "median")
    )
    .reset_index()
)

total_completion_time_kpi["avg_completion_time_seconds"] = total_completion_time_kpi["avg_completion_time_seconds"].round(2)
total_completion_time_kpi["median_completion_time_seconds"] = total_completion_time_kpi["median_completion_time_seconds"].round(2)
total_completion_time_kpi["avg_completion_time_minutes"] = total_completion_time_kpi["avg_completion_time_minutes"].round(2)
total_completion_time_kpi["median_completion_time_minutes"] = total_completion_time_kpi["median_completion_time_minutes"].round(2)

total_completion_time_kpi

,Variation,completed_sessions,avg_completion_time_seconds,median_completion_time_seconds,avg_completion_time_minutes,median_completion_time_minutes
0,Control,14885,391.10,271.0,6.52,4.52
1,Test,17879,374.64,237.0,6.24,3.95


In [18]:
# =========================
# 6C. Total Completion Time - Trimmed 80%
# =========================

q_bounds = (
    completed_sessions_time.groupby("Variation")["completion_time_seconds"]
    .quantile([0.10, 0.90])
    .unstack()
    .reset_index()
    .rename(columns={0.10: "q10", 0.90: "q90"})
)

completed_sessions_time_trimmed = completed_sessions_time.merge(q_bounds, on="Variation", how="left")

completed_sessions_time_trimmed = completed_sessions_time_trimmed[
    completed_sessions_time_trimmed["completion_time_seconds"].between(
        completed_sessions_time_trimmed["q10"],
        completed_sessions_time_trimmed["q90"]
    )
].copy()

completed_sessions_time_trimmed["completion_time_minutes"] = (
    completed_sessions_time_trimmed["completion_time_seconds"] / 60
)

total_completion_time_trimmed_kpi = (
    completed_sessions_time_trimmed.groupby("Variation")
    .agg(
        completed_sessions_included=("visit_id", "nunique"),
        avg_completion_time_seconds_trimmed=("completion_time_seconds", "mean"),
        median_completion_time_seconds_trimmed=("completion_time_seconds", "median"),
        avg_completion_time_minutes_trimmed=("completion_time_minutes", "mean"),
        median_completion_time_minutes_trimmed=("completion_time_minutes", "median")
    )
    .reset_index()
)

total_completion_time_trimmed_kpi["avg_completion_time_seconds_trimmed"] = total_completion_time_trimmed_kpi["avg_completion_time_seconds_trimmed"].round(2)
total_completion_time_trimmed_kpi["median_completion_time_seconds_trimmed"] = total_completion_time_trimmed_kpi["median_completion_time_seconds_trimmed"].round(2)
total_completion_time_trimmed_kpi["avg_completion_time_minutes_trimmed"] = total_completion_time_trimmed_kpi["avg_completion_time_minutes_trimmed"].round(2)
total_completion_time_trimmed_kpi["median_completion_time_minutes_trimmed"] = total_completion_time_trimmed_kpi["median_completion_time_minutes_trimmed"].round(2)

total_completion_time_trimmed_kpi

,Variation,completed_sessions_included,avg_completion_time_seconds_trimmed,median_completion_time_seconds_trimmed,avg_completion_time_minutes_trimmed,median_completion_time_minutes_trimmed
0,Control,11924,314.52,271.0,5.24,4.52
1,Test,14340,284.54,237.0,4.74,3.95


In [19]:
# =========================
# 6D. Final Completion Efficiency Summary
# =========================

completion_efficiency_kpi = completion_rate_kpi.merge(
    total_completion_time_kpi,
    on="Variation",
    how="left"
).merge(
    total_completion_time_trimmed_kpi,
    on="Variation",
    how="left"
)

completion_efficiency_kpi = completion_efficiency_kpi[
    [
        "Variation",
        "total_started_clients",
        "completed_clients",
        "completion_rate_%",
        "avg_completion_time_minutes",
        "median_completion_time_minutes",
        "avg_completion_time_minutes_trimmed",
        "median_completion_time_minutes_trimmed"
    ]
]

completion_efficiency_kpi

,Variation,total_started_clients,completed_clients,completion_rate_%,avg_completion_time_minutes,median_completion_time_minutes,avg_completion_time_minutes_trimmed,median_completion_time_minutes_trimmed
0,Control,23397,15318,65.47,6.52,4.52,5.24,4.52
1,Test,26679,18412,69.01,6.24,3.95,4.74,3.95


In [20]:
# =========================
# 7. Backtracking Rate KPI
# =========================

kpi_df = web_df[web_df["Variation"].isin(["Test", "Control"])].copy()
kpi_df["date_time"] = pd.to_datetime(kpi_df["date_time"])

# Define process order
step_order = {
    "start": 0,
    "step_1": 1,
    "step_2": 2,
    "step_3": 3,
    "confirm": 4
}

kpi_df["step_order"] = kpi_df["process_step"].map(step_order)

# Sort events chronologically within each client/session
kpi_df = kpi_df.sort_values(["Variation", "client_id", "visit_id", "date_time"])

# Previous step within the same session
kpi_df["previous_step_order"] = (
    kpi_df.groupby(["Variation", "client_id", "visit_id"])["step_order"]
    .shift(1)
)

# Backtracking event = current step is earlier than previous step
kpi_df["backtracking_event"] = kpi_df["step_order"] < kpi_df["previous_step_order"]

# Client-level summary
client_backtracking = (
    kpi_df.groupby(["Variation", "client_id"], as_index=False)
    .agg(
        started=("process_step", lambda x: (x == "start").any()),
        backtracked=("backtracking_event", "max"),
        backtracking_events=("backtracking_event", "sum")
    )
)

# Only count backtracking among clients who started
client_backtracking["started_and_backtracked"] = (
    client_backtracking["started"] & client_backtracking["backtracked"]
)

# Final KPI table
backtracking_rate_kpi = (
    client_backtracking.groupby("Variation")
    .agg(
        started_clients=("started", "sum"),
        clients_who_backtracked=("started_and_backtracked", "sum"),
        total_backtracking_events=("backtracking_events", "sum")
    )
    .reset_index()
)

backtracking_rate_kpi["backtracking_rate_%"] = (
    backtracking_rate_kpi["clients_who_backtracked"] / backtracking_rate_kpi["started_clients"] * 100
).round(2)

backtracking_rate_kpi

,Variation,started_clients,clients_who_backtracked,total_backtracking_events,backtracking_rate_%
0,Control,23397,6116,9581,26.14
1,Test,26679,8995,16232,33.72


In [ ]:
# =========================
# 8. Drop-off Rate KPI
# =========================
#This is the inverse of Step completion rate so we might remove it.

kpi_df = web_df[web_df["Variation"].isin(["Test", "Control"])].copy()

step_order = ["start", "step_1", "step_2", "step_3", "confirm"]
variation_order = ["Control", "Test"]

# Client-level step flags: one row per client and Variation
client_steps = (
    kpi_df.groupby(["Variation", "client_id"])["process_step"]
    .apply(set)
    .reset_index(name="steps_reached")
)

for step in step_order:
    client_steps[f"reached_{step}"] = client_steps["steps_reached"].apply(lambda x: step in x)

# Calculate drop-off rates
dropoff_results = []

for current_step, next_step in zip(step_order[:-1], step_order[1:]):
    for variation in variation_order:
        temp = client_steps[client_steps["Variation"] == variation]

        clients_reached_current_step = temp[f"reached_{current_step}"].sum()
        clients_reached_next_step = temp[f"reached_{next_step}"].sum()

        clients_dropped_off = clients_reached_current_step - clients_reached_next_step

        dropoff_rate = (
            clients_dropped_off / clients_reached_current_step
            if clients_reached_current_step > 0
            else np.nan
        )

        dropoff_results.append({
            "Variation": variation,
            "current_step": current_step,
            "next_step": next_step,
            "clients_reached_current_step": clients_reached_current_step,
            "clients_reached_next_step": clients_reached_next_step,
            "clients_dropped_off": clients_dropped_off,
            "dropoff_rate_%": round(dropoff_rate * 100, 2)
        })

dropoff_rate_kpi = pd.DataFrame(dropoff_results)

dropoff_rate_kpi["Variation"] = pd.Categorical(dropoff_rate_kpi["Variation"], categories=variation_order, ordered=True)
dropoff_rate_kpi["current_step"] = pd.Categorical(dropoff_rate_kpi["current_step"], categories=step_order[:-1], ordered=True)

dropoff_rate_kpi = (
    dropoff_rate_kpi
    .sort_values(["current_step", "Variation"])
    .reset_index(drop=True)
)

dropoff_rate_kpi

,Variation,current_step,next_step,clients_reached_current_step,clients_reached_next_step,clients_dropped_off,dropoff_rate_%
0,Control,start,step_1,23397,20152,3245,13.87
1,Test,start,step_1,26679,24267,2412,9.04
2,Control,step_1,step_2,20152,18650,1502,7.45
3,Test,step_1,step_2,24267,22258,2009,8.28
4,Control,step_2,step_3,18650,17422,1228,6.58
5,Test,step_2,step_3,22258,20881,1377,6.19
6,Control,step_3,confirm,17422,15434,1988,11.41
7,Test,step_3,confirm,20881,18687,2194,10.51


In [24]:

demo_df = pd.read_csv("1.1_Clean_Files/df1_demo_clean_plus_experiments_clients.csv")
# =========================
# Client-level Behaviour KPIs
# =========================

kpi_df = web_df[web_df["Variation"].isin(["Test", "Control"])].copy()
kpi_df["date_time"] = pd.to_datetime(kpi_df["date_time"])

# -------------------------
# Completion flags
# -------------------------

client_completion = (
    kpi_df.groupby(["Variation", "client_id"], as_index=False)
    .agg(
        started=("process_step", lambda x: (x == "start").any()),
        completed=("process_step", lambda x: (x == "confirm").any())
    )
)

client_completion["started_and_completed"] = client_completion["started"] & client_completion["completed"]

# -------------------------
# Average iteration time by client
# -------------------------

visit_summary = (
    kpi_df.groupby(["Variation", "client_id", "visit_id"], as_index=False)
    .agg(
        visit_start=("date_time", "min"),
        visit_end=("date_time", "max"),
        step_visits=("process_step", "count")
    )
)

visit_summary["visit_duration_seconds"] = (
    visit_summary["visit_end"] - visit_summary["visit_start"]
).dt.total_seconds()

client_iteration = (
    visit_summary.groupby(["Variation", "client_id"], as_index=False)
    .agg(
        total_time_seconds=("visit_duration_seconds", "sum"),
        total_step_visits=("step_visits", "sum")
    )
)

client_iteration["avg_iteration_time_seconds"] = (
    client_iteration["total_time_seconds"] / client_iteration["total_step_visits"]
)

# -------------------------
# Average number of step visits per client
# -------------------------

client_step_visits = (
    kpi_df.groupby(["Variation", "client_id"], as_index=False)
    .agg(
        step_visits_per_client=("process_step", "count"),
        sessions=("visit_id", "nunique")
    )
)

# -------------------------
# Combine client-level behaviour table
# -------------------------

client_behavior = (
    client_completion
    .merge(client_iteration[["Variation", "client_id", "avg_iteration_time_seconds"]], on=["Variation", "client_id"], how="left")
    .merge(client_step_visits[["Variation", "client_id", "step_visits_per_client", "sessions"]], on=["Variation", "client_id"], how="left")
)

client_behavior.head()

,Variation,client_id,started,completed,started_and_completed,avg_iteration_time_seconds,step_visits_per_client,sessions
0,Control,1028,True,False,False,59.777778,9,1
1,Control,1104,True,False,False,0.000000,2,2
2,Control,1186,True,False,False,5.500000,4,2
3,Control,1195,True,True,True,49.000000,5,1
4,Control,1197,True,True,True,13.571429,7,1


In [25]:
# =========================
# Client Profile Segments
# =========================

demo_base = demo_df[demo_df["Variation"].isin(["Test", "Control"])].copy()

# Gender clean: keep only M and F
demo_base["gender_clean"] = (
    demo_base["gender"]
    .replace({
        "Male": "M",
        "Female": "F",
        "M": "M",
        "F": "F"
    })
)

# Create one row per client
# Median is used because the demo dataset may contain duplicated client_id rows
client_profile = (
    demo_base.groupby(["Variation", "client_id"], as_index=False)
    .agg(
        client_age=("client_age", "median"),
        gender=("gender_clean", lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan),
        client_tenure_years=("client_tenure_years", "median"),
        balance=("balance", "median")
    )
)

# Keep only M and F
client_profile = client_profile[client_profile["gender"].isin(["M", "F"])].copy()

client_profile.head()

,Variation,client_id,client_age,gender,client_tenure_years,balance
0,Control,1195,56.5,M,9.0,43738.68
1,Control,1197,35.5,M,20.0,115749.34
2,Control,3647,24.5,F,13.0,46313.66
5,Control,8101,62.5,F,10.0,35340.59
6,Control,13831,55.5,M,19.0,60679.55


In [26]:
# =========================
# Segment Groups
# =========================

# Age groups in bins of 10 years
age_min = int(np.floor(client_profile["client_age"].min() / 10) * 10)
age_max = int(np.ceil(client_profile["client_age"].max() / 10) * 10)

age_bins = list(range(age_min, age_max + 10, 10))
age_labels = [f"{age_bins[i]}-{age_bins[i+1]-1}" for i in range(len(age_bins)-1)]

client_profile["age_group"] = pd.cut(
    client_profile["client_age"],
    bins=age_bins,
    labels=age_labels,
    right=False,
    include_lowest=True
)

# Tenure groups: 0-10, 10-20, +20 years
client_profile["tenure_group"] = pd.cut(
    client_profile["client_tenure_years"],
    bins=[0, 10, 20, np.inf],
    labels=["0-10 years", "10-20 years", "20+ years"],
    right=False,
    include_lowest=True
)

# Balance groups: under 100K, under 200K, over 200K
client_profile["balance_group"] = pd.cut(
    client_profile["balance"],
    bins=[-np.inf, 100000, 200000, np.inf],
    labels=["Under 100K", "100K-200K", "Over 200K"],
    right=False
)

client_profile.head()

,Variation,client_id,client_age,gender,client_tenure_years,balance,age_group,tenure_group,balance_group
0,Control,1195,56.5,M,9.0,43738.68,50-59,0-10 years,Under 100K
1,Control,1197,35.5,M,20.0,115749.34,30-39,20+ years,100K-200K
2,Control,3647,24.5,F,13.0,46313.66,20-29,10-20 years,Under 100K
5,Control,8101,62.5,F,10.0,35340.59,60-69,10-20 years,Under 100K
6,Control,13831,55.5,M,19.0,60679.55,50-59,10-20 years,Under 100K


In [27]:
# =========================
# Merge Behaviour + Segments
# =========================

client_segment_performance = client_behavior.merge(
    client_profile,
    on=["Variation", "client_id"],
    how="inner"
)

client_segment_performance.head()

,Variation,client_id,started,completed,started_and_completed,avg_iteration_time_seconds,step_visits_per_client,sessions,client_age,gender,client_tenure_years,balance,age_group,tenure_group,balance_group
0,Control,1195,True,True,True,49.000000,5,1,56.5,M,9.0,43738.68,50-59,0-10 years,Under 100K
1,Control,1197,True,True,True,13.571429,7,1,35.5,M,20.0,115749.34,30-39,20+ years,100K-200K
2,Control,3647,True,False,False,134.000000,6,2,24.5,F,13.0,46313.66,20-29,10-20 years,Under 100K
3,Control,8101,True,True,True,190.800000,5,1,62.5,F,10.0,35340.59,60-69,10-20 years,Under 100K
4,Control,13831,True,False,False,266.444444,9,2,55.5,M,19.0,60679.55,50-59,10-20 years,Under 100K


In [28]:
# =========================
# Segment Performance Function
# =========================

def calculate_segment_performance(df, segment_col):
    segment_kpi = (
        df.groupby([segment_col, "Variation"], observed=True)
        .agg(
            clients=("client_id", "nunique"),
            started_clients=("started", "sum"),
            completed_clients=("started_and_completed", "sum"),
            avg_iteration_time_seconds=("avg_iteration_time_seconds", "mean"),
            median_iteration_time_seconds=("avg_iteration_time_seconds", "median"),
            avg_step_visits_per_client=("step_visits_per_client", "mean"),
            median_step_visits_per_client=("step_visits_per_client", "median")
        )
        .reset_index()
    )

    segment_kpi["completion_rate_%"] = (
        segment_kpi["completed_clients"] / segment_kpi["started_clients"] * 100
    ).round(2)

    segment_kpi["avg_iteration_time_seconds"] = segment_kpi["avg_iteration_time_seconds"].round(2)
    segment_kpi["median_iteration_time_seconds"] = segment_kpi["median_iteration_time_seconds"].round(2)
    segment_kpi["avg_step_visits_per_client"] = segment_kpi["avg_step_visits_per_client"].round(2)
    segment_kpi["median_step_visits_per_client"] = segment_kpi["median_step_visits_per_client"].round(2)

    segment_kpi["Variation"] = pd.Categorical(
        segment_kpi["Variation"],
        categories=["Control", "Test"],
        ordered=True
    )

    segment_kpi = segment_kpi.sort_values([segment_col, "Variation"]).reset_index(drop=True)

    return segment_kpi

In [29]:
# =========================
# Segment KPI Tables
# =========================

age_group_performance = calculate_segment_performance(client_segment_performance, "age_group")
gender_performance = calculate_segment_performance(client_segment_performance, "gender")
tenure_group_performance = calculate_segment_performance(client_segment_performance, "tenure_group")
balance_group_performance = calculate_segment_performance(client_segment_performance, "balance_group")

In [30]:
age_group_performance

,age_group,Variation,clients,started_clients,completed_clients,avg_iteration_time_seconds,median_iteration_time_seconds,avg_step_visits_per_client,median_step_visits_per_client,completion_rate_%
0,10-19,Control,5,5,1,63.01,0.00,3.60,1.0,20.00
1,10-19,Test,3,3,0,0.00,0.00,1.00,1.0,0.00
2,20-29,Control,149,149,83,43.11,31.60,4.46,5.0,55.70
3,20-29,Test,161,159,93,37.16,26.29,4.60,5.0,58.49
4,30-39,Control,699,694,475,57.12,42.17,5.70,5.0,68.44
5,30-39,Test,908,892,629,56.73,38.16,6.29,5.0,70.52
6,40-49,Control,1299,1296,927,57.34,43.22,6.81,5.0,71.53
7,40-49,Test,1556,1545,1142,61.62,42.90,7.44,6.0,73.92
8,50-59,Control,1229,1224,908,56.48,41.80,6.33,5.0,74.18
9,50-59,Test,1586,1573,1151,59.32,41.82,6.89,6.0,73.17


In [31]:
gender_performance

,gender,Variation,clients,started_clients,completed_clients,avg_iteration_time_seconds,median_iteration_time_seconds,avg_step_visits_per_client,median_step_visits_per_client,completion_rate_%
0,F,Control,2077,2069,1437,56.22,41.20,6.12,5.0,69.45
1,F,Test,2568,2543,1774,58.11,39.89,6.63,5.0,69.76
2,M,Control,1702,1696,1173,55.65,40.79,6.19,5.0,69.16
3,M,Test,2054,2030,1498,58.09,40.33,6.84,5.0,73.79


In [32]:
tenure_group_performance

,tenure_group,Variation,clients,started_clients,completed_clients,avg_iteration_time_seconds,median_iteration_time_seconds,avg_step_visits_per_client,median_step_visits_per_client,completion_rate_%
0,0-10 years,Control,971,967,687,55.02,40.80,6.04,5.0,71.04
1,0-10 years,Test,1230,1213,861,58.37,38.60,6.54,5.0,70.98
2,10-20 years,Control,2396,2387,1663,57.12,42.15,6.39,5.0,69.67
3,10-20 years,Test,2898,2870,2079,59.07,41.00,6.98,6.0,72.44
4,20+ years,Control,412,411,260,51.49,35.88,5.01,5.0,63.26
5,20+ years,Test,494,490,332,51.75,38.44,5.68,5.0,67.76


In [33]:
balance_group_performance

,balance_group,Variation,clients,started_clients,completed_clients,avg_iteration_time_seconds,median_iteration_time_seconds,avg_step_visits_per_client,median_step_visits_per_client,completion_rate_%
0,Under 100K,Control,2764,2756,1997,57.42,42.00,6.40,5.0,72.46
1,Under 100K,Test,3399,3366,2479,59.19,41.60,7.01,6.0,73.65
2,100K-200K,Control,723,718,485,55.22,41.50,5.81,5.0,67.55
3,100K-200K,Test,947,939,660,58.62,37.60,6.35,5.0,70.29
4,Over 200K,Control,292,291,128,44.03,30.50,4.57,5.0,43.99
5,Over 200K,Test,276,268,133,42.84,30.65,4.49,5.0,49.63
